In [12]:
from __future__ import annotations

import ast
import os
import sys
from pathlib import Path

import numpy as np

REPO = Path.cwd().resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from runners.common import read_session_names_file


In [16]:
pluribus_root = REPO / "pluribus"
ALL_SESSIONS = sorted(f.name for f in os.scandir(pluribus_root) if f.is_dir())
print(len(ALL_SESSIONS), "total session folders under pluribus/")

# Roster must match .phh strings exactly (case-sensitive).
REQUIRED_PLAYERS = frozenset({"Bill", "MrBlue", "Pluribus"})
RNG_SEED = 42


def roster_from_session(session_name: str) -> frozenset[str] | None:
    """Return player names for the first numbered hand in the session, or None if unreadable."""
    session_dir = pluribus_root / session_name
    if not session_dir.is_dir():
        return None
    phh = sorted(
        session_dir.glob("*.phh"),
        key=lambda p: int(p.stem) if p.stem.isdigit() else 0,
    )
    if not phh:
        return None
    for line in phh[0].read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line.startswith("players ="):
            names = ast.literal_eval(line.partition("=")[2].strip())
            return frozenset(names)
    return None

92 total sessions


In [ ]:
train_sessions = set(read_session_names_file(REPO / "sessions_train.txt"))
n_train = len(train_sessions)
if n_train == 0:
    raise ValueError("sessions_train.txt is empty — cannot size theta/filter splits.")

# Counts are relative to the current sessions_train.txt (number of sessions).
n_theta = max(0, round(0.6 * n_train))
n_filter = max(0, round(0.4 * n_train))
need = n_theta + n_filter

eligible: list[str] = []
for name in ALL_SESSIONS:
    if name in train_sessions:
        continue
    roster = roster_from_session(name)
    if roster is None:
        continue
    if REQUIRED_PLAYERS <= roster:
        eligible.append(name)

print(f"sessions_train.txt: {n_train} sessions (held out from theta/filter)")
print(f"target |theta| = round(60% of train) = {n_theta}")
print(f"target |filter| = round(40% of train) = {n_filter}")
print(f"need {need} disjoint sessions total")
print(f"eligible (Bill ∩ MrBlue ∩ Pluribus roster, not in train): {len(eligible)}")

if len(eligible) < need:
    raise RuntimeError(
        f"Need {need} sessions but only {len(eligible)} eligible. "
        "Adjust sessions_train.txt or roster requirements."
    )

In [20]:
rng = np.random.default_rng(RNG_SEED)
perm = rng.permutation(len(eligible)).tolist()
picked = [eligible[i] for i in perm[:need]]

theta_folders = sorted(picked[:n_theta])
filter_folders = sorted(picked[n_theta : n_theta + n_filter])

(REPO / "sessions_theta.txt").write_text("\n".join(theta_folders) + "\n", encoding="utf-8")
(REPO / "sessions_filter.txt").write_text("\n".join(filter_folders) + "\n", encoding="utf-8")

assert not train_sessions.intersection(theta_folders)
assert not train_sessions.intersection(filter_folders)
assert not set(theta_folders).intersection(filter_folders)

print("Wrote sessions_theta.txt:", len(theta_folders), "sessions")
print("Wrote sessions_filter.txt:", len(filter_folders), "sessions")

In [21]:
# Optional: re-verify disjointness and roster after editing sessions_train.txt
_train = set(read_session_names_file(REPO / "sessions_train.txt"))
for label, path in (("theta", REPO / "sessions_theta.txt"), ("filter", REPO / "sessions_filter.txt")):
    names = read_session_names_file(path)
    assert not _train.intersection(names), f"{label} overlaps sessions_train.txt"
    for n in names:
        r = roster_from_session(n)
        assert r is not None and REQUIRED_PLAYERS <= r, (n, r)
print("OK: theta/filter are disjoint from train and satisfy roster filter.")

In [ ]:
# Run cells 0 → 4 top-to-bottom after changing `sessions_train.txt` or `RNG_SEED`.